In [ ]:
import pandas as pd           

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve)
from sklearn.dummy import DummyClassifier    

#visualize
import plotly.express as px
import plotly.graph_objects as go
print("Imported")


In [ ]:
df_acc_feat = pd.read_csv('D:/LendingClub/Datasets/sample_of_accepted_loans_clean_with_features.csv', low_memory=False)

In [ ]:
df_acc_feat.shape
df_acc_feat.head()


,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,purpose,dti,...,pub_rec,revol_util,loan_status,issue_d,target,payment_to_income,loan_to_income,total_interest,has_pub_rec,grade_num
0,3600.0,36.0,13.99,123.03,C,10.0,MORTGAGE,55000.0,debt_consolidation,5.91,...,0.0,29.7,Fully Paid,Dec-2015,0,0.026837053733730826,0.0654533553935383,829.0799999999999,0,2
1,24700.0,36.0,11.99,820.28,C,10.0,MORTGAGE,65000.0,small_business,16.06,...,0.0,19.2,Fully Paid,Dec-2015,0,0.15140835538054512,0.3799941539360933,4830.079999999998,0,2
2,20000.0,60.0,10.78,432.66,B,10.0,MORTGAGE,63000.0,home_improvement,10.78,...,0.0,56.2,Fully Paid,Dec-2015,0,0.08239573414587698,0.31745527848764304,5959.600000000002,0,1
3,10400.0,60.0,22.45,289.91,F,3.0,MORTGAGE,104433.0,major_purchase,25.37,...,0.0,64.5,Fully Paid,Dec-2015,0,0.03330863133706736,0.09958442652775916,6994.600000000002,0,5
4,11950.0,36.0,13.44,405.18,C,4.0,RENT,34000.0,debt_consolidation,10.2,...,0.0,68.4,Fully Paid,Dec-2015,0,0.142954251440668,0.35146025116908325,2636.4799999999996,0,2


---
## Step 8: Data Preprocessing

Before training any model, we need to:
1. **Select features** — which columns to use
2. **Encode categoricals** — convert text to numbers
3. **Split** — train set vs test set (we never touch the test set until final evaluation)
4. **Scale** — make all numeric features have the same scale
5. **Handle imbalance** — use SMOTE to create synthetic minority samples


In [ ]:
# ==========================================================
# Select Features
# ==========================================================
# Features used to train the model
FEATURE_COLS = [
    "loan_amnt",
    "int_rate",
    "installment",
    "term",
    "annual_inc",
    "dti",
    "fico_range_low",
    "emp_length",
    "open_acc",
    "pub_rec",
    "revol_util",

    # Engineered features
    "payment_to_income",
    "loan_to_income",
    "total_interest",
    "has_pub_rec",
    "grade_num"
]

# Keep only columns that exist
FEATURE_COLS = [col for col in FEATURE_COLS if col in df_acc_feat.columns]

print("Selected Features")
print("-" * 30)

for feature in FEATURE_COLS:
    print(f"{feature}")

print(f"\nTotal Features: {len(FEATURE_COLS)}")

Selected Features
------------------------------
loan_amnt
int_rate
installment
term
annual_inc
dti
fico_range_low
emp_length
open_acc
pub_rec
revol_util
payment_to_income
loan_to_income
total_interest
has_pub_rec
grade_num

Total Features: 16


In [ ]:
# Encode Categorical Features
# Categorical columns
categorical_cols = ["home_ownership", "purpose"]

# Keep only existing columns
categorical_cols = [col for col in categorical_cols if col in df_acc_feat.columns]

# Apply One-Hot Encoding
df_encoded = pd.get_dummies(
    df_acc_feat,
    columns=categorical_cols,
    drop_first=True
)

# Add the new dummy variables to the feature list
dummy_cols = [
    col for col in df_encoded.columns
    if col.startswith("home_ownership_") or col.startswith("purpose_")
]

FEATURE_COLS.extend(dummy_cols)
print("Categorical features encoded successfully.")
print(f"Total Features: {len(FEATURE_COLS)}")

Categorical features encoded successfully.
Total Features: 34


In [ ]:
X = df_encoded[FEATURE_COLS]
print("Feature data types:")
print(X.dtypes)

Feature data types:
loan_amnt                     object
int_rate                      object
installment                   object
term                          object
annual_inc                    object
dti                           object
fico_range_low                object
emp_length                    object
open_acc                      object
pub_rec                       object
revol_util                    object
payment_to_income             object
loan_to_income                object
total_interest                object
has_pub_rec                   object
grade_num                     object
home_ownership_MORTGAGE         bool
home_ownership_NONE             bool
home_ownership_OTHER            bool
home_ownership_OWN              bool
home_ownership_RENT             bool
purpose_credit_card             bool
purpose_debt_consolidation      bool
purpose_educational             bool
purpose_home_improvement        bool
purpose_house                   bool
purpose_major_purc

In [ ]:
# Convert columns that should be numbers
numeric_columns = [
    "loan_amnt",
    "int_rate",
    "installment",
    "annual_inc",
    "dti",
    "fico_range_low",
    "open_acc",
    "pub_rec",
    "revol_util",
    "term",
    "emp_length",
    "payment_to_income",
    "loan_to_income",
    "total_interest",
    "has_pub_rec",
    "grade_num"
]

for col in numeric_columns:
    if col in df_encoded.columns:
        df_encoded[col] = pd.to_numeric(
            df_encoded[col],
            errors="coerce"
        )


# Convert target to integer
df_encoded["target"] = df_encoded["target"].astype(int)


# Check data types
print("Data types:")
print(df_encoded[FEATURE_COLS].dtypes)

Data types:
loan_amnt                     float64
int_rate                      float64
installment                   float64
term                          float64
annual_inc                    float64
dti                           float64
fico_range_low                float64
emp_length                    float64
open_acc                      float64
pub_rec                       float64
revol_util                    float64
payment_to_income             float64
loan_to_income                float64
total_interest                float64
has_pub_rec                     int64
grade_num                       int64
home_ownership_MORTGAGE          bool
home_ownership_NONE              bool
home_ownership_OTHER             bool
home_ownership_OWN               bool
home_ownership_RENT              bool
purpose_credit_card              bool
purpose_debt_consolidation       bool
purpose_educational              bool
purpose_home_improvement         bool
purpose_house                    bool


In [ ]:
# Split Data into Training and Testing Sets

# Create numeric target
# 0 = Fully Paid
# 1 = Charged Off

df_encoded["target"] = (
    df_encoded["loan_status"] == "Charged Off"
).astype(int)

# Select features and target
X = df_encoded[FEATURE_COLS]
y = df_encoded["target"]

# Make sure target is numeric
# y = pd.to_numeric(y, errors="coerce")

# Remove rows where target is missing
# valid_rows = y.notna()

# X = X.loc[valid_rows]
# y = y.loc[valid_rows]

# # Convert target to integer
# y = y.astype(int)


# -----------------------------
# 4. Split the data
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print("Dataset Split")
print(f"Training samples : {len(X_train):,}")
print(f"Testing samples  : {len(X_test):,}")

print("-" * 30)
print("\nDefault Rate")
print(f"Training Set : {y_train.mean():0.1%}")
print(f"Testing Set  : {y_test.mean():0.1%}")

Dataset Split
Training samples : 1,076,248
Testing samples  : 269,062
------------------------------

Default Rate
Training Set : 20.0%
Testing Set  : 20.0%


In [ ]:
# Scale Features

scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler on test data
X_test_scaled = scaler.transform(X_test)
print("Feature scaling completed.")
print(
    f"Mean before scaling : "
    f"{X_train.iloc[:, 0].mean():.2f}"
)
print(
    f"Mean after scaling  : "
    f"{X_train_scaled[:, 0].mean():.2f}"
)

Feature scaling completed.
Mean before scaling : 14420.95
Mean after scaling  : 0.00


In [ ]:
print("Non-numeric columns in X_train:")
print(X_train.select_dtypes(exclude="number").columns.tolist())

Non-numeric columns in X_train:
['home_ownership_MORTGAGE', 'home_ownership_NONE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_educational', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_renewable_energy', 'purpose_small_business', 'purpose_vacation', 'purpose_wedding']


In [ ]:
# Balance the Training Data with SMOTE
# Create SMOTE object

smote = SMOTE(random_state=42)

# Apply SMOTE only to the training data
X_train_balanced, y_train_balanced = smote.fit_resample( X_train_scaled, y_train)

print("SMOTE applied successfully\n")

print("Before SMOTE")
print(y_train.value_counts())

print("\nAfter SMOTE")
print(y_train_balanced.value_counts())

SMOTE applied successfully

Before SMOTE
target
0    861401
1    214847
Name: count, dtype: int64

After SMOTE
target
1    861401
0    861401
Name: count, dtype: int64


In [ ]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (1076248, 34)
y_train: (1076248,)
X_test : (269062, 34)
y_test : (269062,)


In [ ]:
print("X_train_scaled:", X_train_scaled.shape)
print("X_train_balanced:", X_train_balanced.shape)


X_train_scaled: (1076248, 34)
X_train_balanced: (1722802, 34)


In [ ]:
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

In [ ]:
# Baseline Model (Dummy Classifier)
# Create and train the baseline model

dummy_model = DummyClassifier(strategy="most_frequent", random_state=42)

dummy_model.fit(X_train_balanced, y_train_balanced)

# Make predictions
y_pred_dummy = dummy_model.predict(X_test_scaled)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred_dummy)
precision = precision_score(y_test, y_pred_dummy, zero_division=0)
recall = recall_score(y_test, y_pred_dummy, zero_division=0)
f1 = f1_score(y_test, y_pred_dummy, zero_division=0)

# Display results
print("\nBaseline Model Results")
print("=" * 35)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")


Baseline Model Results
Accuracy : 0.8004
Precision: 0.0000
Recall   : 0.0000
F1 Score : 0.0000


---
## Step 10 — Model Training & Evaluation

We train 3 different algorithms and compare them.

| Algorithm | How it works (simplified) |
|-----------|--------------------------|
| **Logistic Regression** | Finds a straight line/plane that separates classes. Simple and fast. |
| **Random Forest** | Builds many decision trees and takes a majority vote. |
| **XGBoost** | Builds trees one by one, each fixing the previous one's mistakes. Often the best performer. |


In [ ]:
models = {

    # 1. Logistic Regression
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced",
        solver="liblinear"
    ),

    # 2. Random Forest
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),

    # 3. XGBoost
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )
}

In [ ]:
results = []

for name, model in models.items():

    print(f"Training {name}")

    # Train
    model.fit(X_train_balanced, y_train_balanced)

    # Predict
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # Save results
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "y_prob": y_prob,
        "y_pred": y_pred
    })

Training Logistic Regression
Training Random Forest
Training XGBoost


In [ ]:
# Compare Model Performance

results_df = pd.DataFrame(results)

# Sort by F1 Score (highest first)
results_df = results_df.sort_values(by="F1", ascending=False)
print("Model Comparison")
print("=" * 80)
print(results_df.round(4))

Model Comparison
                 Model  Accuracy  Precision  Recall      F1  ROC-AUC  \
0  Logistic Regression    0.6531     0.3172  0.6402  0.4242   0.7049   
1        Random Forest    0.7691     0.3917  0.2836  0.3290   0.6893   
2              XGBoost    0.7924     0.4530  0.1936  0.2713   0.7030   

                                              y_prob  \
0  [0.32706279486688317, 0.4560764187409845, 0.35...   
1  [0.15, 0.225, 0.03, 0.405, 0.475, 0.39, 0.535,...   
2  [0.23687789, 0.27740142, 0.11906591, 0.3112416...   

                                              y_pred  
0  [0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, ...  
1  [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...  
2  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  


---
## Step 11: Model Evaluation & Validation

Now we compare all 3 models side by side using:
- **Precision**: of all predicted defaults, how many were real?
- **Recall**: of all real defaults, how many did we catch?
- **F1**: harmonic mean of precision & recall
- **ROC-AUC**: overall ability to distinguish defaults from non-defaults


In [ ]:
# Compare Model Performance

# Convert results list to DataFrame
summary_df = pd.DataFrame(results)

# Sort models by F1 Score (highest first)
summary_df = summary_df.sort_values(by="F1", ascending=False)

print("=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
print(summary_df.round(4).to_string(index=False))
print("=" * 80)

# Best model
best_model_name = summary_df.iloc[0]["Model"]

print(f"\nBest Model  : {best_model_name}")
print(f"F1 Score    : {summary_df.iloc[0]['F1']:.4f}")
print(f"ROC-AUC     : {summary_df.iloc[0]['ROC-AUC']:.4f}")

MODEL COMPARISON
              Model  Accuracy  Precision  Recall     F1  ROC-AUC                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
# Model Performance Comparison
# Convert results to DataFrame
results_df = pd.DataFrame(results)

# Select the metrics we want to compare
metrics = ["Precision", "Recall", "F1", "ROC-AUC"]

# Reshape the data
plot_df = results_df.melt(
    id_vars="Model",
    value_vars=metrics,
    var_name="Metric",
    value_name="Score"
)

# Create bar chart
fig = px.bar(
    plot_df,
    x="Metric",
    y="Score",
    color="Model",
    barmode="group",
    text="Score",
    title="Model Performance Comparison",
    color_discrete_sequence=px.colors.qualitative.Set2
)

# Add target line
fig.add_hline(
    y=0.30,
    line_dash="dash",
    annotation_text="Target = 0.30"
)

# Format numbers on bars
fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

# Improve chart appearance
fig.update_layout(
    xaxis_title="Metric",
    yaxis_title="Score",
    yaxis_range=[0, 1.05],
    template="plotly_white",
    width=900,
    height=550
)

fig.show()

In [ ]:
fig = go.Figure()

for result in results:
    fpr, tpr, _ = roc_curve(y_test, result["y_prob"])

    fig.add_trace(
        go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{result['Model']} (AUC={result['ROC-AUC']:.3f})"
        )
    )

# Random classifier
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        line=dict(dash="dash", color="black"),
        name="Random Guess"
    )
)

fig.update_layout(
    title="ROC Curve Comparison",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    template="plotly_white",
    width=800,
    height=600
)

fig.show()

---
## Step 12: Model Interpretation & Insights

A model that we can't explain is hard to trust.  
Feature importance tells us **which inputs the model relies on most**.


In [ ]:
# Feature Importance - Logistic Regression

# Find the best model based on F1 score
best_result = max(results, key=lambda x: x["F1"])

# Get model name
best_model_name = best_result["Model"]

# Get the actual trained model
best_model = models[best_model_name]

print("Best Model:", best_model_name)
print(f"F1 Score: {best_result['F1']:.3f}")


# Logistic Regression Feature Importance

if hasattr(best_model, "coef_"):

    # Create DataFrame
    importance_df = pd.DataFrame({
        "Feature": FEATURE_COLS,
        "Coefficient": best_model.coef_[0]
    })

    # Absolute coefficient
    importance_df["Importance"] = importance_df["Coefficient"].abs()

    # Sort by importance
    importance_df = importance_df.sort_values(
        "Importance",
        ascending=False
    )

    # Select top 15 features
    top_features = importance_df.head(15)

    # ======================================================
    # Create Plotly chart
    # ======================================================

    fig = px.bar(
        top_features.sort_values("Importance"),
        x="Importance",
        y="Feature",
        orientation="h",
        title="Top 15 Features - Logistic Regression",
        labels={
            "Importance": "Absolute Coefficient",
            "Feature": "Feature"
        },
        text="Importance",
        color="Importance",
        color_continuous_scale=px.colors.sequential.Viridis
    )

    fig.update_traces(
        texttemplate="%{text:.3f}",
        textposition="outside"
    )

    fig.update_layout(
        template="plotly_white",
        width=900,
        height=600
    )

    fig.show()

    # Show Top 10 Features

    print("\nTop 10 Features:")
    print(
        importance_df[
            ["Feature", "Coefficient", "Importance"]
        ].head(10).to_string(index=False)
    )


Best Model: Logistic Regression
F1 Score: 0.424



Top 10 Features:
                Feature  Coefficient  Importance
    home_ownership_RENT     0.327771    0.327771
                   term     0.305278    0.305278
              grade_num     0.255611    0.255611
         fico_range_low    -0.254104    0.254104
               int_rate     0.249148    0.249148
             annual_inc    -0.217484    0.217484
         total_interest    -0.170693    0.170693
     home_ownership_OWN     0.168007    0.168007
                    dti     0.166153    0.166153
home_ownership_MORTGAGE     0.161848    0.161848


In [ ]:
# ── Business interpretation of feature importance ─────────
print('='*60)
print('WHAT DOES THE MODEL RELY ON MOST?')
print('='*60)
print()
print('1. INT_RATE / GRADE_NUM')
print('   → Interest rate is the strongest signal.')
print('   → Higher rate = riskier borrower = more likely to default.')
print()
print('2. FICO_RANGE_LOW')
print('   → Lower credit score = worse repayment history = higher risk.')
print()
print('3. DTI (Debt-to-Income Ratio)')
print('   → Borrowers already stretched thin financially default more.')
print()
print('4. PAYMENT_TO_INCOME (engineered)')
print('   → Our new feature! Monthly burden relative to income matters.')
print()
print('5. REVOLVING UTILIZATION')
print('   → High credit card usage signals financial stress.')


WHAT DOES THE MODEL RELY ON MOST?

1. INT_RATE / GRADE_NUM
   → Interest rate is the strongest signal.
   → Higher rate = riskier borrower = more likely to default.

2. FICO_RANGE_LOW
   → Lower credit score = worse repayment history = higher risk.

3. DTI (Debt-to-Income Ratio)
   → Borrowers already stretched thin financially default more.

4. PAYMENT_TO_INCOME (engineered)
   → Our new feature! Monthly burden relative to income matters.

5. REVOLVING UTILIZATION
   → High credit card usage signals financial stress.


---
## 📋 Step 13 — Final Summary & Conclusions


In [ ]:
# ── Answer the 3 business questions ──────────────────────
print('='*65)
print('BUSINESS QUESTIONS — ANSWERED')
print('='*65)
print()
print('Q1: Which borrower characteristics most predict default?')
print('- Interest rate & loan grade (riskier grade = much higher default rate)')
print('- FICO credit score (lower score = higher default risk)')
print('- Debt-to-Income ratio (stretched borrowers default more)')
print('- Revolving credit utilization (high usage = financial stress)')
print()
print('Q2: How do accepted vs rejected applicants differ?')
print('- Rejected applicants have lower FICO scores')
print('- Rejected applicants have higher DTI ratios')
print('- Rejected applicants have lower annual income')
print('- LendingClub\'s rejection criteria align with actual default predictors')
print()
print('Q3: Can we build a model with Precision ≥ 0.3 AND Recall ≥ 0.3?')
best = max(results, key=lambda r: r['F1'])
p_ok = 'YES' if best['Precision'] >= 0.3 else 'NO'
r_ok = 'YES' if best['Recall']    >= 0.3 else 'NO'
print(f'- Best model: {best["Model"]}')
print(f'- Precision {best["Precision"]:.4f} — {p_ok}')
print(f'- Recall    {best["Recall"]:.4f} — {r_ok}')
print()
print('='*65)
print('NEXT STEPS FOR A JUNIOR DATA SCIENTIST')
print('='*65)
print('  1. Try SHAP values for deeper explainability (pip install shap)')
print('  2. Collect more real data — more rows = better model')
print('  3. Experiment with threshold tuning (0.3–0.7) to adjust Precision/Recall trade-off')
print('  4. Deploy best model as a REST API using FastAPI or Flask')
print('  5. Monitor model performance over time (concept drift detection)')


BUSINESS QUESTIONS — ANSWERED

Q1: Which borrower characteristics most predict default?
- Interest rate & loan grade (riskier grade = much higher default rate)
- FICO credit score (lower score = higher default risk)
- Debt-to-Income ratio (stretched borrowers default more)
- Revolving credit utilization (high usage = financial stress)

Q2: How do accepted vs rejected applicants differ?
- Rejected applicants have lower FICO scores
- Rejected applicants have higher DTI ratios
- Rejected applicants have lower annual income
- LendingClub's rejection criteria align with actual default predictors ✅

Q3: Can we build a model with Precision ≥ 0.3 AND Recall ≥ 0.3?
- Best model: Logistic Regression
- Precision 0.3172 — YES
- Recall    0.6402 — YES

NEXT STEPS FOR A JUNIOR DATA SCIENTIST
  1. Try SHAP values for deeper explainability (pip install shap)
  2. Collect more real data — more rows = better model
  3. Experiment with threshold tuning (0.3–0.7) to adjust Precision/Recall trade-off
  4. 